In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)

OUT_DIR = Path("out/audio/")

LABELS = ["MF", "SK", "SJ"]
LABEL_ALIAS = {
    "MF": "MF",
    "SK": "SK",
    "SJ": "Neg"
}
TAU = 0.2

VAR_THRESH = 1e-6


In [ ]:
train = pd.read_csv(OUT_DIR / "windows_train_with_audio.csv")
val   = pd.read_csv(OUT_DIR / "windows_val_with_audio.csv")
test  = pd.read_csv(OUT_DIR / "windows_test_with_audio.csv")

for df in (train, val, test):
    if "w_center_x" in df.columns and "w_center_y" in df.columns:
        df.rename(columns={"w_center_x": "w_center"}, inplace=True)
        df.drop(columns=["w_center_y"], inplace=True)
    elif "w_center_x" in df.columns:
        df.rename(columns={"w_center_x": "w_center"}, inplace=True)
    elif "w_center_y" in df.columns:
        df.rename(columns={"w_center_y": "w_center"}, inplace=True)

print("train/val/test:", len(train), len(val), len(test))


In [ ]:
def make_targets_and_masks(df):
    Y = np.stack([(df[f"y_{lab}"].to_numpy() >= TAU).astype(np.float32) for lab in LABELS], axis=1)
    M = np.stack([df[f"mask_{lab}"].to_numpy().astype(np.float32) for lab in LABELS], axis=1)
    return Y, M

Y_train, M_train = make_targets_and_masks(train)
Y_val,   M_val   = make_targets_and_masks(val)
Y_test,  M_test  = make_targets_and_masks(test)

print("Y_train shape:", Y_train.shape, "M_train shape:", M_train.shape)
print("Train positive rates:", {lab: float(Y_train[:,i].mean()) for i,lab in enumerate(LABELS)})


In [ ]:
DROP = {
    "video_id","w_start","w_end","w_center","split",
    "mask_any","is_bg_negative",
    "y_MF","y_SK","y_SJ","mask_MF","mask_SK","mask_SJ",
}

feat_cols = [c for c in train.columns if c not in DROP and pd.api.types.is_numeric_dtype(train[c])]

print("Raw audio feature columns:", len(feat_cols))
print("Missing-all-features (train):",
      train[[c for c in feat_cols if c != "has_any_audio_feature"]].isna().all(axis=1).mean())

mu = train[feat_cols].mean(skipna=True)
sd = train[feat_cols].std(skipna=True).replace(0, 1.0)

def transform(df):
    X = (df[feat_cols] - mu) / sd
    X = X.fillna(0.0)
    return X

X_train = transform(train)
X_val   = transform(val)
X_test  = transform(test)

var = X_train.var(axis=0)
keep = var[var > VAR_THRESH].index.tolist()

X_train = X_train[keep]
X_val   = X_val[keep]
X_test  = X_test[keep]

print("Kept features after variance filter:", len(keep))
print("X_train:", X_train.shape)


In [ ]:
def masked_metrics(y_true, y_prob, mask, labels=("MF","SK","SJ")):
    pr = {}
    roc = {}
    for i, lab in enumerate(labels):
        m = mask[:, i] > 0.5
        yt = y_true[m, i]
        yp = y_prob[m, i]
        if len(np.unique(yt)) < 2:
            pr[lab] = np.nan
            roc[lab] = np.nan
            continue
        pr[lab] = average_precision_score(yt, yp)
        roc[lab] = roc_auc_score(yt, yp)
    return pr, roc

def fit_logreg_per_label(X, Y, M, labels=("MF","SK","SJ"), seed=42):
    models = {}
    for i, lab in enumerate(labels):
        m = M[:, i] > 0.5
        Xi = X.loc[m].to_numpy()
        yi = Y[m, i].astype(int)

        if len(np.unique(yi)) < 2:
            models[lab] = None
            continue

        classes = np.array([0, 1])
        w = compute_class_weight(class_weight="balanced", classes=classes, y=yi)
        class_weight = {0: w[0], 1: w[1]}

        clf = LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=2000,
            class_weight=class_weight,
            random_state=seed,
        )
        clf.fit(Xi, yi)
        models[lab] = clf
    return models

def predict_proba_per_label(models, X, labels=("MF","SK","SJ")):
    probs = np.zeros((len(X), len(labels)), dtype=np.float32)
    for i, lab in enumerate(labels):
        clf = models.get(lab)
        if clf is None:
            probs[:, i] = np.nan
        else:
            probs[:, i] = clf.predict_proba(X.to_numpy())[:, 1]
    return probs


In [ ]:
def select_cols(prefixes=(), contains=(), exclude_contains=()):
    cols = []
    for c in keep:
        ok = True
        if prefixes:
            ok = ok and any(c.startswith(p) for p in prefixes)
        if contains:
            ok = ok and any(k in c for k in contains)
        if exclude_contains:
            ok = ok and (not any(k in c for k in exclude_contains))
        if ok:
            cols.append(c)
    return cols

GROUPS = {
    "Intensity": select_cols(prefixes=("Loudness",)),
    "Pitch": select_cols(prefixes=("F0semitoneFrom27.5Hz",)),
    "VoiceQuality": select_cols(contains=("jitter", "shimmer", "HNR")),
    "Spectral": select_cols(prefixes=("mfcc", "slope", "alphaRatio", "hammarbergIndex", "spectralFlux")),
}

covered = set(sum(GROUPS.values(), []))
GROUPS["Other"] = [c for c in keep if c not in covered]

print({g: len(cols) for g, cols in GROUPS.items()})


In [ ]:
results_val = []
results_test = []

GROUPS_WITH_ALL = {"All": keep, **GROUPS}

for gname, cols in GROUPS_WITH_ALL.items():
    if len(cols) == 0:
        continue

    models = fit_logreg_per_label(X_train[cols], Y_train, M_train, labels=LABELS, seed=SEED)

    Pva = predict_proba_per_label(models, X_val[cols], labels=LABELS)
    Pte = predict_proba_per_label(models, X_test[cols], labels=LABELS)

    pr_val, roc_val = masked_metrics(Y_val, Pva, M_val, labels=LABELS)
    pr_test, roc_test = masked_metrics(Y_test, Pte, M_test, labels=LABELS)

    for lab in LABELS:
        results_val.append({"group": gname, "label": lab, "PR_AUC": pr_val[lab], "ROC_AUC": roc_val[lab]})
        results_test.append({"group": gname, "label": lab, "PR_AUC": pr_test[lab], "ROC_AUC": roc_test[lab]})

val_tbl = pd.DataFrame(results_val)
test_tbl = pd.DataFrame(results_test)
val_tbl["label_disp"] = val_tbl["label"].map(LABEL_ALIAS)
test_tbl["label_disp"] = test_tbl["label"].map(LABEL_ALIAS)

val_pivot = (
    val_tbl
    .pivot(index="group", columns="label_disp", values="PR_AUC")
    .loc[list(GROUPS_WITH_ALL.keys())]
)

test_pivot = (
    test_tbl
    .pivot(index="group", columns="label_disp", values="PR_AUC")
    .loc[list(GROUPS_WITH_ALL.keys())]
)

print("VAL ablation (PR-AUC):")
display(val_pivot)

print("TEST ablation (PR-AUC):")
display(test_pivot)

plt.figure()
test_pivot.plot(kind="bar")
plt.ylabel("PR-AUC (TEST)")
plt.title("Feature-group ablation (Audio, LogReg)")
plt.legend(title="Label") 
plt.tight_layout()
plt.show()


In [ ]:
res_test = pd.DataFrame(results_test)
print(res_test.columns)
voice_all = res_test[
    (res_test["group"] == "All") 
].copy()

voice_all["label"] = voice_all["label"].map(LABEL_ALIAS)
voice_all


In [ ]:
labels = voice_all["label"].tolist()
pr = voice_all["PR_AUC"].values
roc = voice_all["ROC_AUC"].values

x = np.arange(len(labels))
w = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - w/2, pr, w, label="PR-AUC")
plt.bar(x + w/2, roc, w, label="ROC-AUC")

plt.xticks(x, labels, fontsize=18)
plt.ylabel("Score", fontsize=18)
plt.title("Voice Modality Baseline (openSMILE eGeMAPS)", fontsize=14)
plt.ylim(0, 1)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.legend()

for i, v in enumerate(pr):
    plt.text(i - w/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=10)

for i, v in enumerate(roc):
    plt.text(i + w/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=10)

plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.text(-0.4, 0.51, "Chance", fontsize=14, color="gray")
plt.tight_layout()
plt.savefig("voice_baseline_pr_roc.png", dpi=300)
plt.show()


In [ ]:
logreg_all = fit_logreg_per_label(X_train[keep], Y_train, M_train, labels=LABELS, seed=SEED)

coef_rows = []
for lab in LABELS:
    clf = logreg_all.get(lab)
    if clf is None:
        continue

    coef = pd.Series(clf.coef_[0], index=keep)
    abscoef = coef.abs()

    for gname, cols in GROUPS.items():
        if len(cols) == 0:
            continue
        coef_rows.append({
            "label": lab,
            "group": gname,
            "mean_abs_coef": float(abscoef[cols].mean()),
            "sum_abs_coef": float(abscoef[cols].sum()),
            "n_features": len(cols),
        })

coef_df = pd.DataFrame(coef_rows)

display(coef_df.sort_values(["label","mean_abs_coef"], ascending=[True, False]))

for lab in LABELS:
    sub = coef_df[coef_df["label"] == lab].sort_values("mean_abs_coef", ascending=False)
    plt.figure()
    plt.bar(sub["group"], sub["mean_abs_coef"])
    plt.ylabel("Mean |coefficient|")
    plt.title(f"Coefficient magnitude by feature group (LogReg, {lab})")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
for lab in LABELS:
    clf = logreg_all.get(lab)
    if clf is None:
        continue
    coef = pd.Series(clf.coef_[0], index=keep)
    top = coef.reindex(coef.abs().sort_values(ascending=False).index).head(15)
    print(f"\nTop 15 features by |coef| for {lab}:")
    display(top.to_frame("coef"))


In [ ]:
df = test.copy()

df["MF_hard"] = (df["y_MF"] >= TAU).astype(int)
df["SK_hard"] = (df["y_SK"] >= TAU).astype(int)
df["SJ_hard"] = (df["y_SJ"] >= TAU).astype(int)
df["ANY_hard"] = ((df[["MF_hard","SK_hard","SJ_hard"]].sum(axis=1) > 0)).astype(int)

loud_col = "Loudness_sma3_mean"
pitch_col = "F0semitoneFrom27.5Hz_sma3nz_mean"

print("Has loudness:", loud_col in df.columns, "| Has pitch:", pitch_col in df.columns)

def boxplot_feature_vs_label(df_in, feature_col, label_col, title):
    d = df_in[[feature_col, label_col]].dropna()
    if d.empty:
        return
    plt.figure()
    plt.boxplot(
        [d[d[label_col]==0][feature_col].values, d[d[label_col]==1][feature_col].values],
        labels=[f"{label_col}=0", f"{label_col}=1"]
    )
    plt.ylabel(feature_col)
    plt.title(title)
    plt.tight_layout()
    plt.show()

for lab in ["MF_hard", "SK_hard", "SJ_hard", "ANY_hard"]:
    boxplot_feature_vs_label(df, loud_col, lab, f"{loud_col} vs {lab} (TEST)")

for lab in ["MF_hard", "SK_hard", "SJ_hard", "ANY_hard"]:
    boxplot_feature_vs_label(df, pitch_col, lab, f"{pitch_col} vs {lab} (TEST)")

df_voiced = df[df["has_any_audio_feature"] == 1].copy()
for lab in ["MF_hard", "SK_hard", "SJ_hard", "ANY_hard"]:
    boxplot_feature_vs_label(df_voiced, pitch_col, lab, f"{pitch_col} vs {lab} (TEST, voiced windows)")
